### Attempting to use r.water.outlet from Grass for delineating the upstream catchment of multiple points in Jamaica

In [ ]:
# launch this notebook from a GRASS GIS terminal

from pathlib import Path

from grass.script.core import gisenv
import grass.script as gs
import grass.jupyter as gj
import geopandas as gpd
import pandas as pd

In [ ]:
# gs.create_project("nbs_project", epsg=3448)

In [ ]:
session = gj.init("nbs_project")

In [ ]:
gisenv()

In [ ]:
drainage_tif = "/Users/robynhaggis/Documents/Geospatial_analysis/drainage_direction_100.tif"   # r.watershed drainage-direction raster (EPSG:3448)
# points_gpkg  = "/Users/robynhaggis/Documents/rp1500_points_with_stream.gpkg"
# points_layer = "rp1500_points_with_stream"                                  # layer name inside the GPKG

In [ ]:
gs.run_command("r.import", input=drainage_tif, output="drainage_direction_100")

In [ ]:
gs.run_command("g.region", raster="drainage_direction_100")

In [ ]:
out_dir = Path("/Users/robynhaggis/Documents/Geospatial_analysis/upstream_basins_100")
out_dir.mkdir(parents=True, exist_ok=True)

example_coords = (661594.566, 708177.874)  # easting, northing from point snapped to stream network
example_out_tif = out_dir / "test.tif"
example_out_gpkg = out_dir / "test.gpkg"

In [ ]:
gs.run_command("r.water.outlet", input="drainage_direction_100", output="example_output", coordinates=example_coords)

In [ ]:
# gs.run_command("r.out.gdal", input="example_output", output=str(example_out_tif), format="GTiff", createopt="COMPRESSION=LZW")

In [ ]:
gs.run_command("r.to.vect", input="example_output", output="example_output_vector",type="area")

In [ ]:
gs.run_command("v.out.ogr", input="example_output_vector", output=str(example_out_gpkg), format="GPKG")

In [ ]:
! ls -alh {out_dir}

In [ ]:
# points_to_catchment_df = pd.read_parquet(
#     "/Users/robynhaggis/Documents/Geospatial_analysis/points_to_catchment.parquet"
# )
path = "/Users/robynhaggis/Documents/Geospatial_analysis/points_to_catchment.parquet"
df = gpd.read_parquet(path)
# points_to_catchment_df

In [ ]:
# type(points_to_catchment_df)

In [ ]:
for row in df[["dem_i","dem_j","geometry"]].itertuples(index=False, name="Row"):
    print(row.geometry.x, row.geometry.y)
    print(row.dem_i, row.dem_j)
    break

In [ ]:
# # this cell for the other notebook

# for p in points_to_catchment_df.itertuples():
#     print(p.geometry_y.x, p.geometry_y.y) # easting, northing to get coordinates for GRASS r.water.outlet
#     print(p.dem_i, p.dem_j) # dem i and j to use in output name and filename (and later, reading in again here)
#     break

In [ ]:
for dem_i, dem_j, geom in gdf[["dem_i", "dem_j", "geometry"]].itertuples(index=False, name=None):
    x, y = float(geom.x), float(geom.y)
    base      = f"basin_{dem_i}_{dem_j}"
    vect_map  = f"{base}_vec"
    rast_tif  = out_dir / f"{base}.tif"
    vect_gpkg = out_dir / f"{base}.gpkg"

    try:
        gs.run_command(
            "r.water.outlet",
            input="drainage_direction_100",
            output=base,
            coordinates=(x, y),
            overwrite=True,
        )

        gs.run_command(
            "r.to.vect",
            input=base,
            output=vect_map,
            type="area",
            overwrite=True,
        )

        gs.run_command(
            "v.out.ogr",
            input=vect_map,
            output=str(vect_gpkg),
            format="GPKG",
            overwrite=True,
        )

        gs.run_command(
            "r.out.gdal",
            input=base,
            output=str(rast_tif),
            format="GTiff",
            createopt="COMPRESS=LZW",
            nodata=0,
            overwrite=True,
        )

        print(f"Wrote {vect_gpkg.name} and {rast_tif.name}")

    except Exception as e:
        print(f"Failed for ({dem_i},{dem_j}) at ({x},{y}): {e}")